# September 16, 2026 calculation companion

S2 confirms at +2.04 percentage points; S3 is contra at +1.5 points. The registered joint confirmation is false. The video's $800B rental estimate is a full-deployment annual rate; uniform H2 commissioning gives $200B during that calendar year under the same price assumption.

This notebook uses frozen public observations and the existing signal functions. It makes no network calls and requires only Python's standard library. Sources and reporting dates are in the adjacent JSON file. Prices stop at September 15; reported NAV is June 30. Thresholds and basket membership remain unchanged.

The source fetches and all Python cells were executed. Jupyter kernel execution and rendering were not checked; nbformat, nbclient and ipykernel are absent from the environment. The read-only reproduction command from the repository root is `python3 scripts/verify_refresh_2026_09_16.py`.


In [1]:
from pathlib import Path
from datetime import date
import hashlib
import json
import runpy
import statistics

root = Path.cwd()
if not (root / 'src/bubble/market_signals.py').exists():
    root = root.parent
data = json.loads((root / 'analysis/refresh_2026-09-16.json').read_text())
market = data['market']
module = root / data['calculation_source']['path']
assert hashlib.sha256(module.read_bytes()).hexdigest() == data['calculation_source']['sha256'], 'Signal implementation changed; review reproducibility.'
signals = runpy.run_path(str(module))
assert market['price_cutoff'] == '2026-09-15'
print('Loaded frozen September 15 market observations; June 30 NAVs; unchanged registered rules.')


Loaded frozen September 15 market observations; June 30 NAVs; unchanged registered rules.


Credit spreads are option-adjusted spreads, in percentage points. Year-to-date changes use the first nonmissing 2026 observation, following the registered implementation. The complete captured series is preserved.


In [2]:
series_to_key = {'BAMLH0A0HYM2': 'hy_oas', 'BAMLH0A3HYC': 'ccc_oas', 'BAMLH0A1HYBB': 'bb_oas'}
credit = {}
for series, key in series_to_key.items():
    evidence = market['fred_evidence'][series]
    rows = evidence['observations']
    assert rows == sorted(rows)
    assert len({row[0] for row in rows}) == len(rows)
    assert rows[0] == evidence['baseline'] and rows[-1] == evidence['latest']
    assert rows[-1][0] <= market['price_cutoff']
    credit[key] = {'value': rows[-1][1], 'date': rows[-1][0], 'ytd_chg': round(rows[-1][1] - rows[0][1], 2)}
assert credit == market['credit']
s2 = signals['eval_s2'](credit)
assert s2 == market['s2']
print(json.dumps(s2, indent=2))


{
  "id": "S2_ccc_divergence",
  "status": "confirming",
  "ccc_minus_hy_ytd_pp": 2.04,
  "ccc_oas": 10.85,
  "bb_ytd_chg_pp": -0.1,
  "market_wide_flag": false,
  "desc": "(CCC-HY) OAS differential +2.04pp YTD; >=1.5pp = tail-specific repricing (confirming), <=0pp = contra"
}


Discount equals 100 * (1 - unadjusted closing price / reported NAV). Negative discounts are premiums. The fixed exposed set is BXSL/ARCC; controls are MAIN/GBDC/TSLX/PSEC. OBDC evidence has changed, so this remains a historical-basket comparison with incomplete current exposure coverage.


In [3]:
bdc = {}
for ticker, record in market['bdc'].items():
    prices = [row for row in record['recent_raw_price_observations'] if row[0] <= market['price_cutoff']]
    chosen = max(prices, key=lambda row: row[0])
    assert chosen[0] == record['date'] and round(chosen[1], 2) == record['close']
    assert record['nav_asof'] == '2026-06-30' and record['nav'] > 0
    discount = round(100 * (1 - record['close'] / record['nav']), 1)
    assert discount == record['discount_pct']
    bdc[ticker] = {'discount_pct': discount}
s3 = signals['eval_s3'](bdc)
assert s3 == market['s3']
confirm2 = s2['status'] == 'confirming' and not s2['market_wide_flag'] and s3['status'] == 'confirming'
assert confirm2 == market['compound_confirm2']
print(json.dumps(s3, indent=2))
print('CONFIRM-2:', confirm2)


{
  "id": "S3_bdc_discount_differential",
  "status": "contra",
  "worst": "BXSL",
  "worst_discount_pct": 2.9,
  "control_median_pct": 1.4,
  "differential_pp": 1.5,
  "desc": "worst AI-exposed BDC discount (BXSL 2.9%) minus non-AI control median (1.4%) = +1.5pp; >=15pp = exposure-specific stress (confirming), <=5pp = inside normal sector dispersion (contra)"
}
CONFIRM-2: False


Scenario arithmetic: capacity is GW, rental price is millions of dollars per MW-year, and outputs are billions of dollars. Uniform H2 commissioning averages one quarter of the final incremental capacity over the calendar year. These examples exclude existing capacity and pre-service reservation payments. They are not revenue forecasts.


In [4]:
assumptions = data['video_assumptions']
run_rates = [assumptions['inference_gw_at_full_deployment'] * x for x in assumptions['lab_revenue_usd_b_per_gw_year']]
assert run_rates == [750, 1500]
print('Inference full-deployment annual revenue rates, USD billions:', run_rates)
for gw in assumptions['incremental_capacity_gw']:
    annual_rate = gw * 1000 * assumptions['rental_usd_m_per_mw_year'] / 1000
    for timing, fraction in assumptions['arrival_average_fraction_of_year'].items():
        calendar_revenue = annual_rate * fraction
        print(f'{gw} GW, {timing}: calendar rental {calendar_revenue:g}B; full-deployment annual rate {annual_rate:g}B')
provider_unit_price = assumptions['monthly_provider_payment_usd_b'] * 12 / assumptions['capacity_for_video_unit_price_gw']
print(f'Illustrative provider receipts / lab expense: {provider_unit_price:g}B per GW-year; not lab end-customer revenue.')


Inference full-deployment annual revenue rates, USD billions: [750, 1500]
30 GW, all_online_january_1: calendar rental 600B; full-deployment annual rate 600B
30 GW, uniform_full_year: calendar rental 300B; full-deployment annual rate 600B
30 GW, uniform_second_half: calendar rental 150B; full-deployment annual rate 600B
40 GW, all_online_january_1: calendar rental 800B; full-deployment annual rate 800B
40 GW, uniform_full_year: calendar rental 400B; full-deployment annual rate 800B
40 GW, uniform_second_half: calendar rental 200B; full-deployment annual rate 800B
Illustrative provider receipts / lab expense: 50B per GW-year; not lab end-customer revenue.


The public filing check records a bounded search, not proof that a confidential document or an unindexed filing does not exist. Stale issuance cannot count as an open funding window.


In [5]:
check = data['anthropic_public_s1_check']
hits = check['edgar_response']['hits']
assert hits['total']['relation'] == 'eq'
assert hits['total']['value'] == len(hits['hits']) == 5
issuers = sorted({name for hit in hits['hits'] for name in hit['_source']['display_names']})
assert all('ANTHROPIC' not in issuer.upper() for issuer in issuers)
print('EDGAR query hits:', len(hits['hits']), 'Issuer names:', issuers)
as_of = date.fromisoformat(data['as_of'])
for key in ['original_latest_card_date', 'galaxy_pricing_date']:
    age = (as_of - date.fromisoformat(data['issuance_freshness'][key])).days
    print(f'{key}: {age} days; over registered 45-day freshness limit: {age > signals["STALE_DAYS"]}')


EDGAR query hits: 5 Issuer names: ['Gloo Holdings, Inc.  (GLOO)  (CIK 0002069785)', 'QumulusAI, Inc.  (QMLS)  (CIK 0002084026)', 'SPACE EXPLORATION TECHNOLOGIES CORP  (SPCX)  (CIK 0001181412)']
original_latest_card_date: 106 days; over registered 45-day freshness limit: True
galaxy_pricing_date: 55 days; over registered 45-day freshness limit: True


The calculations preserve the original prediction rules. They do not establish a global capacity inventory, audited lab profitability or an AI-caused credit event. The accompanying synthesis records source limitations and corrections to older research.
